# Средние токены prompt / ответа по моделям

Считает **оценку** числа токенов по сохранённым CSV `results_*.csv`: полный промпт восстанавливается через `utils.build_prompt3` (как в прогоне), выход — колонка `raw_output` (или `json`, если `raw_output` нет). Строки с **пустым ответом** в расчёт средних/медиан не входят. **В каждой папке с результатами** (например `…/qwen-3-8b/DETAILED_INSTR_…/`) берётся **только последняя по дате** версия `results_*.csv` (суффикс `_YYYYMMDD_HHMM` в имени; иначе по времени файла).

Имя шаблона промпта подхватывается из соседнего `metrics_*.json`, если он есть; иначе задаётся `DEFAULT_PROMPT_TEMPLATE` ниже. Устаревшие имена из метрик (напр. `DETAILED_INSTR_ZEROSHOT_BASELINE_OUTLINES_RUS`) приводятся к реальным константам в `prompt_config` (суффиксы `_OUTLINES`/`_GUIDANCE`, алиасы, имя папки прогона).

Токенизатор: `transformers.AutoTokenizer` по **`model_name`** из `metrics_*.json`, если это валидный HuggingFace repo id. Для Ollama в метриках часто лежит тег вида `codegemma:7b-...` (с двоеточием) — тогда берётся HF id из **`model_key`** + `models.yaml` или по имени файла CSV.

Загрузка токенизатора сначала **только из локального HF-кэша** (`local_files_only=True`), чтобы не дергать hub без нужды (иначе возможен таймаут). Если модели ещё нет в кэше — выполняется обычная загрузка из сети.

In [34]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

import pandas as pd
import yaml
from transformers import AutoTokenizer

# Корень проекта: запускайте ячейки из корня репозитория (где лежит models.yaml)
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models.yaml").exists():
    raise FileNotFoundError(f"Не найден models.yaml — смените cwd на корень проекта. Сейчас: {PROJECT_ROOT}")

RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_YAML = PROJECT_ROOT / "models.yaml"

# Если рядом с CSV нет metrics_*.json — используется этот шаблон (см. config.PROMPT_TEMPLATE_NAME)
DEFAULT_PROMPT_TEMPLATE = "DETAILED_INSTR_ZEROSHOT_BASELINE"

os.chdir(PROJECT_ROOT)
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import build_prompt3

In [35]:
def load_hf_model_ids() -> dict[str, str]:
    """Ключ модели из models.yaml -> HuggingFace id (поле name)."""
    with open(MODELS_YAML, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    out = {}
    for key, spec in (cfg.get("models") or {}).items():
        name = (spec or {}).get("name")
        if name:
            out[key] = name
    return out


def normalize_key(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def resolve_hf_id(csv_stem: str, hf_map: dict[str, str]) -> str | None:
    """
    Имя файла вида results_<model_name>_<timestamp> — подбираем HF id.
    """
    m = re.match(r"results_(.+)_(\d{8}_\d{6})\s*$", csv_stem)
    if not m:
        m = re.match(r"results_(.+)$", csv_stem)
        if not m:
            return None
        rest = m.group(1)
    else:
        rest = m.group(1)
    rest_norm = normalize_key(rest)
    for key, hid in hf_map.items():
        if normalize_key(key) == rest_norm:
            return hid
        if normalize_key(hid.split("/")[-1]) == rest_norm:
            return hid
    # Часто rest уже похож на org_model (google_gemma-2-2b-it -> google/gemma-2-2b-it)
    if "_" in rest:
        parts = rest.split("_", 1)
        if len(parts) == 2 and parts[0] and parts[1]:
            guess = f"{parts[0]}/{parts[1].replace('_', '-')}"
            return guess
    return None


def _hf_id_from_model_key(model_key: str, hf_map: dict[str, str]) -> str | None:
    """Ключ из metrics (часто совпадает с ключом в models.yaml)."""
    k = (model_key or "").strip()
    if not k:
        return None
    if k in hf_map:
        return hf_map[k]
    for suf in ("-ollama", "-vllm"):
        if k.endswith(suf):
            base = k[: -len(suf)].strip("_")
            if base in hf_map:
                return hf_map[base]
    return None


def _is_hf_hub_repo_id(s: str | None) -> bool:
    """True, если строка похожа на id репозитория HF (не тег Ollama вида name:tag)."""
    if not s or not isinstance(s, str):
        return False
    t = s.strip()
    if ":" in t:
        return False
    try:
        from huggingface_hub.utils import validate_repo_id

        validate_repo_id(t)
        return True
    except Exception:
        return False


def resolve_tokenizer_hf_id(mb: dict, csv_stem: str, hf_map: dict[str, str]) -> tuple[str | None, str | None]:
    """
    Id для AutoTokenizer: из metrics.model_name, если это валидный HF repo id;
    иначе model_key -> models.yaml; иначе эвристика по имени CSV.
    Возвращает (hf_id или None, сырое model_name из metrics).
    """
    raw_name = mb.get("model_name")
    if raw_name and _is_hf_hub_repo_id(str(raw_name)):
        return str(raw_name).strip(), raw_name if isinstance(raw_name, str) else str(raw_name)
    via_key = _hf_id_from_model_key(mb.get("model_key") or "", hf_map)
    if via_key:
        return via_key, raw_name if isinstance(raw_name, str) else raw_name
    via_csv = resolve_hf_id(csv_stem, hf_map)
    return via_csv, raw_name if isinstance(raw_name, str) else raw_name


def find_metrics_json(csv_path: Path) -> Path | None:
    d = csv_path.parent
    stem = csv_path.stem
    if stem.startswith("results_"):
        mid = stem[len("results_") :]
        cand = d / f"metrics_{mid}.json"
        if cand.is_file():
            return cand
    metrics = sorted(d.glob("metrics_*.json"))
    return metrics[-1] if metrics else None


def load_metrics_bundle(path: Path | None) -> dict:
    if not path or not path.is_file():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    hp = data.get("hyperparameters") or {}
    return {
        "model_name": data.get("model_name"),
        "model_key": (data.get("model_key") or "") or "",
        "prompt_template": hp.get("prompt_template_name")
        or data.get("prompt_template")
        or data.get("effective_prompt_template_name"),
    }


import prompt_config as _prompt_config

# Исторические/«методические» имена из метрик/Sheets, которых нет как атрибутов в prompt_config
_PROMPT_NAME_ALIASES = {
    "DETAILED_INSTR_ZEROSHOT_BASELINE_OUTLINES_RUS": "DETAILED_INSTR_ZEROSHOT_CD_RUS",
    "DETAILED_INSTR_ONESHOT_OUTLINES_RUS": "DETAILED_INSTR_ONESHOT_CD_RUS",
}


def _resolve_single_prompt_name(name: str) -> str | None:
    """Имя из metrics / папки -> атрибут prompt_config или None."""
    n = name.strip()
    if hasattr(_prompt_config, n):
        return n
    if n in _PROMPT_NAME_ALIASES:
        a = _PROMPT_NAME_ALIASES[n]
        if hasattr(_prompt_config, a):
            return a
    if n.endswith("_OUTLINES"):
        s = n[: -len("_OUTLINES")]
        if hasattr(_prompt_config, s):
            return s
    if n.endswith("_GUIDANCE"):
        s = n[: -len("_GUIDANCE")]
        if hasattr(_prompt_config, s):
            return s
    return None


def resolve_prompt_template_name(metrics_raw: str | None, csv_path: Path) -> str:
    """
    Сопоставляет строку из metrics с реальным именем шаблона в prompt_config.
    Подсказка: имя родительской папки (напр. ...DETAILED_INSTR_..._OUTLINES или OLLAMA_..._DETAILED_INSTR_...).
    """
    folder = csv_path.parent.name
    m = re.search(r"(DETAILED_INSTR_[A-Z0-9_]+|MINIMAL_INSTR_[A-Z0-9_]+)", folder)
    folder_hint = m.group(1) if m else None
    for src in (metrics_raw, folder_hint):
        if not src:
            continue
        resolved = _resolve_single_prompt_name(str(src))
        if resolved:
            return resolved
    return DEFAULT_PROMPT_TEMPLATE


_tokenizer_cache: dict[str, object] = {}


def get_tokenizer(hf_id: str):
    """
    Сначала только HF-кэш (local_files_only=True) — без лишних запросов к hub
    (у новых transformers иначе может вызываться list_repo_templates → таймаут без сети).
    Если модели нет в кэше — вторая попытка с загрузкой из сети.
    """
    if hf_id not in _tokenizer_cache:
        try:
            _tokenizer_cache[hf_id] = AutoTokenizer.from_pretrained(
                hf_id,
                trust_remote_code=True,
                local_files_only=True,
            )
        except Exception:
            _tokenizer_cache[hf_id] = AutoTokenizer.from_pretrained(
                hf_id,
                trust_remote_code=True,
                local_files_only=False,
            )
    return _tokenizer_cache[hf_id]


def count_tokens(tok, text: str) -> int:
    if not isinstance(text, str) or not text:
        return 0
    return len(tok.encode(text, add_special_tokens=False))

In [36]:
def summarize_csv(csv_path: Path, hf_map: dict[str, str]) -> dict | None:
    metrics_path = find_metrics_json(csv_path)
    mb = load_metrics_bundle(metrics_path)
    hf_id, metrics_model_name_raw = resolve_tokenizer_hf_id(mb, csv_path.stem, hf_map)
    if not hf_id:
        return {
            "csv": str(csv_path),
            "error": "Не удалось получить HuggingFace id для токенизатора (model_key/models.yaml/имя CSV)",
        }
    raw_pt = mb.get("prompt_template")
    pt = resolve_prompt_template_name(raw_pt, csv_path)
    model_key = mb.get("model_key") or ""

    try:
        df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    except pd.errors.EmptyDataError:
        return {"csv": str(csv_path), "error": "Пустой CSV (нет колонок/данных)"}
    if len(df) == 0:
        return {"csv": str(csv_path), "error": "CSV без строк данных"}
    if "text" not in df.columns:
        return {"csv": str(csv_path), "error": "Нет колонки text"}

    out_col = "raw_output" if "raw_output" in df.columns else ("json" if "json" in df.columns else None)
    if not out_col:
        return {"csv": str(csv_path), "error": "Нет колонок raw_output/json"}

    tok = get_tokenizer(hf_id)
    inp_counts = []
    out_counts = []
    n_total = len(df)
    for _, row in df.iterrows():
        text = row["text"]
        out_text = row[out_col] if out_col else ""
        if not str(out_text).strip():
            continue
        if "prompt" in df.columns and str(row.get("prompt", "")).strip():
            prompt = str(row["prompt"])
        else:
            prompt = build_prompt3(
                text,
                structured_output=False,
                response_schema=None,
                prompt_template_name=pt,
                model_key=model_key or None,
            )
        inp_counts.append(count_tokens(tok, prompt))
        out_counts.append(count_tokens(tok, out_text))

    n = len(inp_counts)
    if n == 0:
        return {
            "csv": str(csv_path.relative_to(PROJECT_ROOT)) if csv_path.is_relative_to(PROJECT_ROOT) else str(csv_path),
            "error": f"Нет строк с непустым ответом ({out_col}); всего строк в CSV: {n_total}",
        }
    skipped = n_total - n
    return {
        "csv": str(csv_path.relative_to(PROJECT_ROOT)) if csv_path.is_relative_to(PROJECT_ROOT) else str(csv_path),
        "model_key": model_key or None,
        "metrics_model_name_raw": metrics_model_name_raw,
        "hf_model_id": hf_id,
        "prompt_template": pt,
        "prompt_template_raw": raw_pt,
        "rows": n,
        "rows_in_csv": n_total,
        "skipped_empty_output": skipped,
        "avg_prompt_tokens": sum(inp_counts) / n,
        "avg_output_tokens": sum(out_counts) / n,
        "median_prompt_tokens": float(pd.Series(inp_counts).median()),
        "median_output_tokens": float(pd.Series(out_counts).median()),
    }


def _results_csv_sort_key(p: Path) -> tuple:
    """Ключ сортировки: больше = новее. Сначала суффикс _YYYYMMDD_HHMM в имени, иначе mtime."""
    stem = p.stem
    m = re.search(r"_(\d{8})_(\d{6})$", stem)
    if m:
        return (2, int(m.group(1)), int(m.group(2)))
    m2 = re.search(r"_(\d{8})$", stem)
    if m2:
        return (1, int(m2.group(1)), 0)
    try:
        return (0, p.stat().st_mtime)
    except OSError:
        return (0, 0.0)


def collect_results_csvs(root: Path) -> list[Path]:
    """
    По одному файлу на папку: в каждой директории с результатами берётся только
    самый новый `results_*.csv` (по дате в суффиксе имени; если её нет — по времени файла).
    """
    all_paths = list(root.rglob("results_*.csv"))
    by_parent: dict[Path, list[Path]] = {}
    for p in all_paths:
        by_parent.setdefault(p.parent, []).append(p)
    chosen = [max(paths, key=_results_csv_sort_key) for paths in by_parent.values()]
    return sorted(chosen)

In [37]:
hf_map = load_hf_model_ids()
csvs = collect_results_csvs(RESULTS_DIR)
print(
    f"CSV для разбора: {len(csvs)} (по одному самому новому results_*.csv в каждой папке) в {RESULTS_DIR}"
)

rows = []
for p in csvs:
    r = summarize_csv(p, hf_map)
    if r and "error" not in r:
        rows.append(r)
    elif r:
        print("SKIP", r.get("csv"), "—", r.get("error"))

summary = pd.DataFrame(rows)
if len(summary):
    summary = summary.sort_values("hf_model_id")
    display(summary)
else:
    print("Нет успешно обработанных файлов. Проверьте путь results/ и models.yaml.")

CSV для разбора: 154 (по одному самому новому results_*.csv в каждой папке) в C:\D\DZ\Masters\Diploma\SmallLLMEvaluator\results


,csv,model_key,metrics_model_name_raw,hf_model_id,prompt_template,prompt_template_raw,rows,rows_in_csv,skipped_empty_output,avg_prompt_tokens,avg_output_tokens,median_prompt_tokens,median_output_tokens
109,results\qwen-2.5-3b\VLLM_Q4_DETAILED_INSTR_ZER...,qwen-2.5-3b,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,DETAILED_INSTR_ZEROSHOT_BASELINE,DETAILED_INSTR_ZEROSHOT_BASELINE,100,100,0,1649.70,327.17,1583.5,267.5
95,results\qwen-2.5-3b\DETAILED_INSTR_ONESHOT\res...,qwen-2.5-3b,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,DETAILED_INSTR_ONESHOT,DETAILED_INSTR_ONESHOT,100,100,0,1918.70,1483.25,1852.5,847.5
107,results\qwen-2.5-3b\MINIMAL_INSTR_FIVESHOT_CD_...,qwen-2.5-3b,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,MINIMAL_INSTR_FIVESHOT_CD_RUS,MINIMAL_INSTR_FIVESHOT_CD_RUS,100,100,0,2364.70,542.00,2298.5,542.0
106,results\qwen-2.5-3b\MINIMAL_INSTR_FIVESHOT_CD_...,qwen-2.5-3b,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,MINIMAL_INSTR_FIVESHOT_CD_RUS,MINIMAL_INSTR_FIVESHOT_CD_RUS,100,100,0,2364.70,187.30,2298.5,138.5
105,results\qwen-2.5-3b\MINIMAL_INSTR_FIVESHOT\res...,qwen-2.5-3b,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,MINIMAL_INSTR_FIVESHOT,MINIMAL_INSTR_FIVESHOT,100,100,0,2384.70,953.81,2318.5,1024.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,results\mistral-3-8b-instruct\MINIMAL_INSTR_FI...,mistral-3-8b-instruct,mistralai/Ministral-3-8B-Instruct-2512,mistralai/Ministral-3-8B-Instruct-2512,MINIMAL_INSTR_FIVESHOT,MINIMAL_INSTR_FIVESHOT,100,100,0,2635.82,245.60,2561.0,168.5
92,results\mistral-3-8b-instruct\MINIMAL_INSTR_FI...,mistral-3-8b-instruct,mistralai/Ministral-3-8B-Instruct-2512,mistralai/Ministral-3-8B-Instruct-2512,MINIMAL_INSTR_FIVESHOT_CD_RUS,MINIMAL_INSTR_FIVESHOT_CD_RUS,100,100,0,2617.82,265.11,2543.0,196.0
93,results\mistral-3-8b-instruct\MINIMAL_INSTR_FI...,mistral-3-8b-instruct,mistralai/Ministral-3-8B-Instruct-2512,mistralai/Ministral-3-8B-Instruct-2512,MINIMAL_INSTR_FIVESHOT_CD_RUS,MINIMAL_INSTR_FIVESHOT_CD_RUS,100,100,0,2617.82,271.92,2543.0,205.5
82,results\mistral-3-8b-instruct\DETAILED_INSTR_O...,mistral-3-8b-instruct,mistralai/Ministral-3-8B-Instruct-2512,mistralai/Ministral-3-8B-Instruct-2512,DETAILED_INSTR_ONESHOT_CD_RUS,DETAILED_INSTR_ONESHOT_CD_RUS,100,100,0,2939.41,200.87,2864.0,157.0


### Итоговые средние (по всем успешно обработанным CSV)

Средние **взвешены по числу строк** в каждом файле (`rows`), чтобы прогоны с 100 примерами не считались равными прогону с 10.

In [38]:
try:
    s = summary
except NameError:
    s = None

if s is None or len(s) == 0:
    print("Нет данных: сначала выполните ячейку со сбором summary.")
else:
    n = pd.to_numeric(s["rows"], errors="coerce").fillna(0)
    w = float(n.sum())
    if w <= 0:
        print("Сумма строк по файлам = 0.")
    else:
        ap = pd.to_numeric(s["avg_prompt_tokens"], errors="coerce")
        ao = pd.to_numeric(s["avg_output_tokens"], errors="coerce")
        mean_in = float((ap * n).sum() / w)
        mean_out = float((ao * n).sum() / w)
        totals = pd.DataFrame(
            [
                {
                    "файлов_в_summary": len(s),
                    "всего_строк_примеров": int(w),
                    "среднее_in_токенов_на_пример": round(mean_in, 2),
                    "среднее_out_токенов_на_пример": round(mean_out, 2),
                }
            ]
        )
        display(totals)
        print(
            f"Итого: в среднем ~{mean_in:.1f} токенов на промпт и ~{mean_out:.1f} токенов на ответ на один пример (по {int(w)} строкам в {len(s)} файлах)."
        )

,файлов_в_summary,всего_строк_примеров,среднее_in_токенов_на_пример,среднее_out_токенов_на_пример
0,154,15137,1874.27,301.08


Итого: в среднем ~1874.3 токенов на промпт и ~301.1 токенов на ответ на один пример (по 15137 строкам в 154 файлах).
